# 01 — OffsetsDB exploration

**Goal of this notebook:** pull the real, free CarbonPlan OffsetsDB archive and understand what's actually in it before we design anything downstream.

OffsetsDB consolidates project and credit data from seven registries (Verra, Gold Standard, American Carbon Registry, Climate Action Reserve, ART TREES, Cercarbono, Isometric). It gives us project characteristics, issuances, and retirements — **it does not contain prices**. We'll pull prices separately from the Carbonmark API in notebook 02.

Data source: https://github.com/carbonplan/offsets-db-data (MIT licensed code, data under CarbonPlan's Terms of Data Access — see the archive's `TERMS_OF_DATA_ACCESS.txt`).

Run the cells top to bottom. Each one prints something — read the output before moving to the next cell, that's how you'll actually learn what's in this dataset.

## Step 1 — Download the OffsetsDB archive

This downloads a ~8.5 MB zip directly from CarbonPlan's public S3 bucket. No API key, no account, no package installs needed — just `requests` and `zipfile`, both already in Colab.

In [1]:
import requests
import zipfile
import io
import os

# The official, current download link for the latest OffsetsDB CSV archive
OFFSETSDB_URL = "https://carbonplan-offsets-db.s3.us-west-2.amazonaws.com/production/latest/offsets-db.csv.zip"

# Where we'll keep the raw, untouched data (matches our repo's data/raw convention)
RAW_DIR = "data/raw"
os.makedirs(RAW_DIR, exist_ok=True)

print("Downloading OffsetsDB archive...")
response = requests.get(OFFSETSDB_URL)
response.raise_for_status()  # stops the notebook here with a clear error if the download fails, instead of continuing with garbage
print(f"Downloaded {len(response.content):,} bytes")

# Unzip directly into data/raw
with zipfile.ZipFile(io.BytesIO(response.content)) as z:
    print("Files in archive:", z.namelist())
    z.extractall(RAW_DIR)

print(f"\nExtracted to {RAW_DIR}/")

Downloaded 8,534,671 bytes
Files in archive: ['TERMS_OF_DATA_ACCESS.txt', 'metadata.json', 'credits.csv', 'projects.csv']



Extracted to data/raw/


## Step 2 — Load into pandas

The archive contains two files we care about:
- `projects.csv` — one row per carbon offset project (registry, country, project type, status, etc.)
- `credits.csv` — one row per issuance or retirement transaction, linked to a project by `project_id`

There's also a `TERMS_OF_DATA_ACCESS.txt` and a `metadata.json` telling us when the snapshot was generated — worth checking so we know how fresh the data is.

In [2]:
import pandas as pd
import json

# Check how fresh this snapshot is
with open(f"{RAW_DIR}/metadata.json") as f:
    metadata = json.load(f)
print("Snapshot generated at:", metadata["generated_at"])

projects_df = pd.read_csv(f"{RAW_DIR}/projects.csv")
credits_df = pd.read_csv(f"{RAW_DIR}/credits.csv")

print(f"\nprojects.csv: {projects_df.shape[0]:,} rows, {projects_df.shape[1]} columns")
print(f"credits.csv:  {credits_df.shape[0]:,} rows, {credits_df.shape[1]} columns")

Snapshot generated at: 2026-06-01T20:33:13.820968+00:00


/tmp/ipykernel_564/3929193235.py:10: DtypeWarning: Columns (0: retirement_account, 1: retirement_reason, 2: transaction_url) have mixed types. Specify dtype option on import or set low_memory=False.
  credits_df = pd.read_csv(f"{RAW_DIR}/credits.csv")



projects.csv: 11,659 rows, 18 columns
credits.csv:  532,799 rows, 11 columns


## Step 3 — Explore the projects table

This is the table that will eventually feed our hedonic pricing model's features (registry, project type, country, vintage era). Let's see what we're actually working with.

In [3]:
print("Columns:", list(projects_df.columns))
print()
projects_df.head(3)

Columns: ['category', 'country', 'first_issuance_at', 'first_retirement_at', 'is_compliance', 'issued', 'listed_at', 'name', 'project_id', 'project_type', 'project_type_source', 'project_url', 'proponent', 'protocol', 'protocol_unassigned', 'registry', 'retired', 'status']



,category,country,first_issuance_at,first_retirement_at,is_compliance,issued,listed_at,name,project_id,project_type,project_type_source,project_url,proponent,protocol,protocol_unassigned,registry,retired,status
0,unknown,India,NaN,NaN,False,0.0,NaN,Solar PV Power Project by Juniper Green Ray Tw...,VCS6032,Unknown,carbonplan,https://registry.verra.org/app/projectDetail/V...,Juniper Green Ray Two Private Limited,NaN,['Methodology Under Development'],verra,0.0,listed
1,ghg-management,Indonesia,NaN,NaN,False,0.0,NaN,Recovery and Avoidance of Methane from Industr...,VCS6027,Unknown,carbonplan,https://registry.verra.org/app/projectDetail/V...,KNOWLEDGE INTEGRATION SERVICES SINGAPORE PTE. ...,['ams-iii-h'],NaN,verra,0.0,listed
2,ghg-management,China,NaN,NaN,False,0.0,NaN,Ningxia Junfeng Siguquan Coal Mine Well NO.2 C...,VCS6019,Unknown,carbonplan,https://registry.verra.org/app/projectDetail/V...,"Ningxia Junfeng New Energy Technology Co., Ltd.",['acm0008'],NaN,verra,0.0,listed


In [4]:
# How many projects per registry? This tells us how registry-balanced our sample is.
print("Projects by registry:")
print(projects_df["registry"].value_counts())

Projects by registry:
registry
verra                       4983
gold-standard               4105
climate-action-reserve      1268
american-carbon-registry     979
cercarbono                   234
isometric                     62
art-trees                     28
Name: count, dtype: int64


In [5]:
# Project status matters — a 'listed' project might not have issued credits yet,
# which affects whether it's meaningful to compare its price to an active project.
print("Projects by status:")
print(projects_df["status"].value_counts())

Projects by status:
status
unknown       4650
listed        2878
registered    2241
completed     1745
active         136
canceled         6
inactive         2
Name: count, dtype: int64


In [6]:
# Top 15 project types and top 15 countries — gives a sense of category imbalance
# we'll need to handle later (e.g. if 40% of projects are one type, comparisons need care).
print("Top 15 project types:")
print(projects_df["project_type"].value_counts().head(15))
print()
print("Top 15 countries:")
print(projects_df["country"].value_counts().head(15))

Top 15 project types:
project_type
Cookstove                        1605
Improved Forest Management       1095
Wind                             1021
Unknown                          1007
Afforestation + Reforestation     692
Hydropower                        532
Clean Water                       483
Manure Biodigester                466
Landfill                          442
REDD+                             429
Centralized Solar                 349
Rice Emission                     313
Ozone Depleting Substances        305
Biomass                           279
Sustainable Agriculture           265
Name: count, dtype: int64

Top 15 countries:
country
United States    1909
India            1903
China            1563
Türkiye           574
Mexico            564
Brazil            415
Kenya             346
Uganda            296
Colombia          270
Rwanda            225
Bangladesh        178
South Africa      173
Malawi            158
Nepal             149
Vietnam           147
Name: count,

In [7]:
# Missing data check — which columns have gaps we'll need to handle in cleaning later?
missing = projects_df.isna().sum().sort_values(ascending=False)
missing_pct = (missing / len(projects_df) * 100).round(1)
pd.DataFrame({"missing_count": missing, "missing_pct": missing_pct})

,missing_count,missing_pct
protocol_unassigned,11454,98.2
listed_at,10394,89.2
first_retirement_at,6675,57.3
first_issuance_at,5538,47.5
protocol,1773,15.2
proponent,984,8.4
name,2,0.0
country,1,0.0
status,1,0.0
category,0,0.0


## Step 4 — Explore the credits table

Each row here is one issuance or retirement event for a project. This is what lets us compute how many credits a project has issued vs. retired — a liquidity/activity signal we'll use later.

In [8]:
print("Columns:", list(credits_df.columns))
print()
credits_df.head(3)

Columns: ['project_id', 'quantity', 'retirement_account', 'retirement_beneficiary', 'retirement_beneficiary_harmonized', 'retirement_note', 'retirement_reason', 'transaction_date', 'transaction_type', 'transaction_url', 'vintage']



,project_id,quantity,retirement_account,retirement_beneficiary,retirement_beneficiary_harmonized,retirement_note,retirement_reason,transaction_date,transaction_type,transaction_url,vintage
0,VCS1,12630.0,NaN,NaN,NaN,NaN,NaN,2009-03-26 00:00:00+00:00,issuance,NaN,2007
1,VCS1,9074.0,NaN,NaN,NaN,NaN,NaN,2014-01-21 00:00:00+00:00,issuance,NaN,2006
2,VCS10,153460.0,NaN,NaN,NaN,NaN,NaN,2009-04-22 00:00:00+00:00,issuance,NaN,2006


In [9]:
# Issuance vs retirement split
print("Transaction types:")
print(credits_df["transaction_type"].value_counts())
print()

# Vintage range — the year the emission reduction actually happened
print("Vintage range:", credits_df["vintage"].min(), "to", credits_df["vintage"].max())

Transaction types:
transaction_type
retirement      495403
issuance         37307
cancellation        89
Name: count, dtype: int64

Vintage range: 1996 to 2061


## Step 5 — Sanity checks

Before trusting this data for modelling, two quick checks: do project IDs actually line up between the two tables, and are there any duplicate project records?

In [10]:
# Duplicate project_id check — each project should appear exactly once in projects.csv
duplicate_projects = projects_df["project_id"].duplicated().sum()
print(f"Duplicate project_id rows in projects.csv: {duplicate_projects}")

# Do all credits reference a project_id that actually exists in projects.csv?
credit_project_ids = set(credits_df["project_id"])
known_project_ids = set(projects_df["project_id"])
orphaned = credit_project_ids - known_project_ids

print(f"Unique project_ids referenced in credits.csv: {len(credit_project_ids):,}")
print(f"Of those, {len(orphaned):,} have no matching row in projects.csv")
if orphaned:
    print("Example orphaned IDs:", list(orphaned)[:5])

Duplicate project_id rows in projects.csv: 0


Unique project_ids referenced in credits.csv: 6,133
Of those, 0 have no matching row in projects.csv


## Findings & next steps

Record what you actually found by running the cells above — this becomes the first entry in our research decisions log. Update the summary below with your real numbers before moving on.

**What OffsetsDB gives us:** project-level characteristics (registry, country, project type, status) and a full issuance/retirement transaction history per project.

**What it does not give us:** any price data at all. That's expected — OffsetsDB is a registry-data product, not a market-data product. Prices come from the Carbonmark API in the next notebook.

**Known data quality issues to watch for** (from CarbonPlan's own documentation, confirmed above): a meaningful share of projects have `status = unknown`, and project_type coverage is uneven across registries — both will need explicit handling in the cleaning step rather than being silently dropped.

**Next notebook (02):** pull live listing prices from the Carbonmark API for a sample of these project_ids, so we have something to actually join against this table.

## Step 6 — Save a checkpoint

Save what we've loaded to `data/interim` so notebook 02 can pick up from here without re-downloading.

In [11]:
INTERIM_DIR = "data/interim"
os.makedirs(INTERIM_DIR, exist_ok=True)

projects_df.to_csv(f"{INTERIM_DIR}/projects_raw_loaded.csv", index=False)
credits_df.to_csv(f"{INTERIM_DIR}/credits_raw_loaded.csv", index=False)

print("Saved checkpoint files to", INTERIM_DIR)
print("\nNote: in Colab, /content is wiped when the runtime disconnects.")
print("If you want this to persist across sessions, mount Google Drive:")
print("  from google.colab import drive")
print("  drive.mount('/content/drive')")
print("...and point RAW_DIR / INTERIM_DIR at a path inside /content/drive/MyDrive/ instead.")

Saved checkpoint files to data/interim

Note: in Colab, /content is wiped when the runtime disconnects.
If you want this to persist across sessions, mount Google Drive:
  from google.colab import drive
  drive.mount('/content/drive')
...and point RAW_DIR / INTERIM_DIR at a path inside /content/drive/MyDrive/ instead.
